# SumIterative, RecursiveSum e RecursiveSumLinear (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/SumIterative_RecursiveSum_RecursiveSumLinear_cpp.ipynb)

Capitolo 17 — Algoritmi ricorsivi. Notebook eseguibile di `SumIterative`, `RecursiveSum` e `RecursiveSumLinear`: gli algoritmi in C++, eseguiti sugli esempi del capitolo, con la cella «Prova tu» per provarli con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/17-algoritmi-ricorsivi/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/SumIterative_RecursiveSum_RecursiveSumLinear_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi/SumIterative_RecursiveSum_RecursiveSumLinear_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap17_algoritmi_ricorsivi_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `RecursiveSum` → `RecursiveSum` (PascalCase).

## Somma di un array: SumIterative, RecursiveSum, RecursiveSumLinear

- **SumIterative**$(a,n)$: un ciclo che accumula le somme parziali. $S(n)=3n+4$, memoria extra $\Theta(1)$.
- **RecursiveSum**$(a,l,r)$: divide et impera sulle due metà (caso base $l=r$). $2n-1$ chiamate, $S(n)=7n-5$, profondità $\lceil\log_2 n\rceil+1$.
- **RecursiveSumLinear**$(a,n)$: decremento di uno: $S(n)=3n-1$, profondità $n$.

*Input*: un array di $n\ge1$ interi (per `RecursiveSum`, due indici $l\le r$). *Output*: la somma degli elementi. I contatori `steps` (righe di pseudocodice eseguite), `calls` e `maxDepth` (chiamate contemporaneamente attive) misurano le grandezze analizzate nel capitolo; in `RecursiveSum` e `RecursiveSumLinear` la `return` è unica per poter aggiornare la profondità.

Sull'array dell'esempio $a=[9,8,12,3,5,14,6]$ ($n=7$) la somma è $57$; `RecursiveSum` fa $13$ chiamate e $44$ passi, con profondità $4$. Poi si esegue la traccia della tabella del capitolo su $a=[9,8,12,3,5]$: la colonna *stack* riporta le chiamate attive.

In [1]:
%%writefile sums.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long steps = 0;   // passi eseguiti (righe di pseudocodice)
int calls = 0;         // chiamate eseguite
int depth = 0;         // chiamate attive in questo momento
int maxDepth = 0;      // profondità massima della ricorsione

void Reset()
{
    steps = 0;
    calls = 0;
    depth = 0;
    maxDepth = 0;
}

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Somma iterativa di a[0..n-1]
int SumIterative(int n, int *a)
{
    int s = 0;
    int i = 0;
    steps += 2;                 // righe 1-2: s := 0, i := 1
    while (i < n)
    {
        steps += 3;             // test vero + due righe del corpo
        s = s + a[i];
        i = i + 1;
    }
    steps += 2;                 // test falso + return
    return s;
}

// Somma di a[l..r] (indici da 0), l <= r, per divide et impera
int RecursiveSum(int *a, int l, int r)
{
    calls++;
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    steps++;                    // riga 1: if l = r
    int result;
    if (l == r)
    {
        steps++;                // riga 2: return a[l] (caso base)
        result = a[l];
    }
    else
    {
        steps += 4;             // righe 3-6: m, due chiamate, return
        int m = l + (r - l) / 2;                // divide
        int s1 = RecursiveSum(a, l, m);         // conquer (metà sinistra)
        int s2 = RecursiveSum(a, m + 1, r);     // conquer (metà destra)
        result = s1 + s2;                       // combine
    }
    depth--;
    return result;
}

// Somma di a[0..n-1] per decremento di uno, n >= 1
int RecursiveSumLinear(int *a, int n)
{
    calls++;
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    steps++;                    // riga 1: if n = 1
    int result;
    if (n == 1)
    {
        steps++;                // riga 2: return a[1]
        result = a[0];
    }
    else
    {
        steps += 2;             // righe 3-4: chiamata e return
        int s = RecursiveSumLinear(a, n - 1);
        result = s + a[n - 1];
    }
    depth--;
    return result;
}

// ---- traccia di RecursiveSum ----
int row = 0;
int maxStack = 0;

string Join(const vector<string> &v)
{
    string s;
    for (size_t i = 0; i < v.size(); i++)
    {
        if (i > 0) s += ",";
        s += v[i];
    }
    return s;
}

void Emit(const vector<string> &stack, const string &action)
{
    row++;
    if ((int)stack.size() > maxStack) maxStack = stack.size();
    printf("%2d | %s | %s\n", row, Join(stack).c_str(), action.c_str());
}

string P(int l, int r) { return "(" + to_string(l) + "," + to_string(r) + ")"; }

int RecursiveSumTrace(int *a, int l, int r, vector<string> &stack)
{
    stack.push_back(P(l, r));
    if (l == r)
    {
        Emit(stack, "l=r => return a[" + to_string(l) + "]=" + to_string(a[l]));
        stack.pop_back();
        return a[l];
    }
    int m = l + (r - l) / 2;
    Emit(stack, "l!=r, m=" + to_string(l) + "+(" + to_string(r) + "-" + to_string(l) + ")/2=" + to_string(m) + "; call " + P(l, m));
    int s1 = RecursiveSumTrace(a, l, m, stack);
    Emit(stack, "s1=" + to_string(s1) + "; call " + P(m + 1, r));
    int s2 = RecursiveSumTrace(a, m + 1, r, stack);
    Emit(stack, "s2=" + to_string(s2) + "; return " + to_string(s1) + "+" + to_string(s2) + "=" + to_string(s1 + s2));
    stack.pop_back();
    return s1 + s2;
}

#ifndef NOMAIN
int main()
{
    int a[] = {9, 8, 12, 3, 5, 14, 6};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    Reset();
    int s = SumIterative(n, a);
    printf("SumIterative       = %d   steps=%lld (3n+4=%d)\n", s, steps, 3 * n + 4);
    Reset();
    s = RecursiveSum(a, 0, n - 1);
    printf("RecursiveSum       = %d   calls=%d (2n-1=%d)  steps=%lld (7n-5=%d)  maxDepth=%d\n", s, calls, 2 * n - 1, steps, 7 * n - 5, maxDepth);
    Reset();
    s = RecursiveSumLinear(a, n);
    printf("RecursiveSumLinear = %d   calls=%d  steps=%lld (3n-1=%d)  maxDepth=%d\n", s, calls, steps, 3 * n - 1, maxDepth);

    int b[] = {9, 8, 12, 3, 5};
    printf("\n");
    printf("a = %s   RecursiveSum(a, 0, 4)\n", Fmt(b, 5).c_str());
    printf(" # | stack | azione\n");
    vector<string> stack;
    int res = RecursiveSumTrace(b, 0, 4, stack);
    printf("risultato %d, righe %d, massimo numero di chiamate attive %d\n", res, row, maxStack);
    return 0;
}
#endif

Writing sums.cpp


In [2]:
!g++ -std=c++17 -O2 -o sums sums.cpp && ./sums

a = [9, 8, 12, 3, 5, 14, 6]   n = 7
SumIterative       = 57   steps=25 (3n+4=25)
RecursiveSum       = 57   calls=13 (2n-1=13)  steps=44 (7n-5=44)  maxDepth=4
RecursiveSumLinear = 57   calls=7  steps=20 (3n-1=20)  maxDepth=7

a = [9, 8, 12, 3, 5]   RecursiveSum(a, 0, 4)
 # | stack | azione
 1 | (0,4) | l!=r, m=0+(4-0)/2=2; call (0,2)
 2 | (0,4),(0,2) | l!=r, m=0+(2-0)/2=1; call (0,1)
 3 | (0,4),(0,2),(0,1) | l!=r, m=0+(1-0)/2=0; call (0,0)
 4 | (0,4),(0,2),(0,1),(0,0) | l=r => return a[0]=9
 5 | (0,4),(0,2),(0,1) | s1=9; call (1,1)
 6 | (0,4),(0,2),(0,1),(1,1) | l=r => return a[1]=8
 7 | (0,4),(0,2),(0,1) | s2=8; return 9+8=17
 8 | (0,4),(0,2) | s1=17; call (2,2)
 9 | (0,4),(0,2),(2,2) | l=r => return a[2]=12
10 | (0,4),(0,2) | s2=12; return 17+12=29
11 | (0,4) | s1=29; call (3,4)
12 | (0,4),(3,4) | l!=r, m=3+(4-3)/2=3; call (3,3)
13 | (0,4),(3,4),(3,3) | l=r => return a[3]=3
14 | (0,4),(3,4) | s1=3; call (4,4)
15 | (0,4),(3,4),(4,4) | l=r => return a[4]=5
16 | (0,4),(3,4) | s2=5; retur

**Prova tu.** Cambia l'array `a` (anche la lunghezza) e confronta passi, chiamate e profondità: solo `RecursiveSumLinear` ha profondità $n$.

In [3]:
%%writefile prova_sums.cpp
#define NOMAIN
#include "sums.cpp"

int main()
{
    // Cambia l'array: confronta passi, chiamate e profondità dei tre algoritmi.
    int a[] = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16};
    int n = 16;
    Reset(); int s = SumIterative(n, a);
    printf("%d %lld\n", s, steps);
    Reset(); s = RecursiveSum(a, 0, n - 1);
    printf("%d %lld %d %d\n", s, steps, calls, maxDepth);
    Reset(); s = RecursiveSumLinear(a, n);
    printf("%d %lld %d %d\n", s, steps, calls, maxDepth);
    return 0;
}

Writing prova_sums.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_sums prova_sums.cpp && ./prova_sums

136 52
136 107 31 5
136 47 16 16


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).